<a href="https://colab.research.google.com/github/waghmodedevidas121-cloud/PARAM/blob/main/Ditto_Avatar_Colab_Gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎭 Open Avatar Studio — Ditto + Gradio

**एका फोटोला speech audio देऊन expressive talking-avatar video तयार करा.**  
One portrait image + uploaded/recorded speech → a downloadable MP4 with lip sync, facial expression, eye motion, and head movement.

- **Avatar model:** [Ant Group Ditto](https://github.com/antgroup/ditto-talkinghead), PyTorch checkpoint
- **UI:** Gradio public link
- **Target runtime:** Google Colab **T4 / L4 / A100** GPU
- **License:** Apache‑2.0 (review all dependency licenses for production)
- **No API key required** and no paid avatar API is used

> **Responsible use:** Use only a face and voice you own or have explicit permission to animate. Do not impersonate, deceive, defraud, harass, or create non-consensual media. Keep the disclosure watermark enabled where appropriate.


## Why this model? (checked 20 August 2026)

“Newest” and “works reliably in a normal Colab” are not the same thing.

| Model | Strength | Colab reality |
|---|---|---|
| **LongCat‑Video‑Avatar 1.5** (2026) | New high-end full-body, multi-person generation | Official avatar examples use two GPU processes; not a normal single-T4 notebook target |
| **EchoMimicV3‑Flash** (2026) | 8-step, up to 768², audio-driven body/portrait | Recent and strong, but the full official Python stack has very large base checkpoints and needs high host RAM/offloading |
| **Ditto PyTorch** (selected) | Single photo/audio, expression + head motion, controllable and efficient | Maintainer reports about **3 GB VRAM**; official PyTorch weights avoid fragile TensorRT engine compatibility on T4 |
| **MuseTalk 1.5** | Very fast, accurate lip-region replacement | Excellent for dubbing, but narrower motion generation than Ditto |

This notebook therefore uses the **recent, practical Colab option**, Ditto's open PyTorch backend. It deliberately does **not** use the provided Ampere-only TensorRT engines, so a free-tier T4 can run it.

Sources: [Ditto repository](https://github.com/antgroup/ditto-talkinghead), [Ditto VRAM answer](https://github.com/antgroup/ditto-talkinghead/issues/20), [EchoMimicV3](https://github.com/antgroup/echomimic_v3), [LongCat‑Video](https://github.com/meituan-longcat/LongCat-Video), [MuseTalk](https://github.com/TMElyralab/MuseTalk).


## 0 · Select a GPU runtime

In Colab choose **Runtime → Change runtime type → T4 GPU** (or L4/A100), then run every cell in order.

The setup uses an isolated **Python 3.10** environment, so it does not depend on Colab's changing system-Python version. Expect roughly **3 GB model download** plus the isolated environment.


In [ ]:
import shutil, subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU found. Select Runtime → Change runtime type → T4 GPU, then reconnect.")

subprocess.run([
    "nvidia-smi",
    "--query-gpu=name,memory.total,driver_version",
    "--format=csv,noheader",
], check=True)

usage = shutil.disk_usage("/content")
print(f"Free disk: {usage.free / 1024**3:.1f} GiB")
if usage.free < 9 * 1024**3:
    raise RuntimeError("At least 9 GiB free disk is recommended. Use Runtime → Disconnect and delete runtime.")


## 1 · Install the pinned environment

This installs only the **PyTorch** path—no TensorRT build. The first run can take several minutes.


In [ ]:
import os, subprocess, sys
from pathlib import Path

ENV = Path("/content/ditto-env")
PYTHON = ENV / "bin/python"

def shell(command):
    print("+", command)
    subprocess.run(["bash", "-lc", command], check=True)

shell("apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1 build-essential python3-dev fonts-dejavu-core")
shell(f"{sys.executable} -m pip install -q 'uv>=0.8,<1'")
shell(f"{sys.executable} -m uv venv --python 3.10 --clear {ENV}")

# Match Ditto's published PyTorch/CUDA environment.
shell(
    f"{sys.executable} -m uv pip install --python {PYTHON} "
    "torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121"
)

packages = [
    "numpy==2.0.1",
    "onnxruntime-gpu==1.20.2",
    "librosa==0.10.2.post1",
    "soundfile==0.13.0",
    "opencv-python-headless==4.10.0.84",
    "scikit-image==0.25.0",
    "scipy==1.15.0",
    "numba==0.60.0",
    "llvmlite==0.43.0",
    "imageio==2.36.1",
    "imageio-ffmpeg==0.5.1",
    "filetype==1.2.0",
    "Cython==3.0.11",
    "einops==0.8.1",
    "tqdm==4.67.1",
    "Pillow==11.0.0",
    "setuptools==75.1.0",
    "protobuf>=4.25.3,<5",
    "absl-py",
    "attrs",
    "flatbuffers",
    "matplotlib",
    "sounddevice",
    "sentencepiece",
    "gradio==5.44.1",
    "huggingface-hub==0.34.4",
    "hf-xet>=1.1.5,<2",
    "psutil",
]
quoted = " ".join(repr(item) for item in packages)
shell(f"{sys.executable} -m uv pip install --python {PYTHON} {quoted}")

# 0.10.14 is the official MediaPipe wheel used without forcing NumPy back to 1.x.
shell(f"{sys.executable} -m uv pip install --python {PYTHON} --no-deps mediapipe==0.10.14")
print("Environment ready:", PYTHON)


## 2 · Download pinned Ditto source

Pinning the upstream commit keeps the notebook reproducible even if `main` changes later.


In [ ]:
import shutil, subprocess
from pathlib import Path

ROOT = Path("/content/ditto-talkinghead")
COMMIT = "c3e47eee2e626500017a0556b470d6d4182f85e8"

if ROOT.exists() and not (ROOT / ".git").exists():
    shutil.rmtree(ROOT)

if not (ROOT / ".git").exists():
    subprocess.run([
        "git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/antgroup/ditto-talkinghead.git", str(ROOT),
    ], check=True)

subprocess.run(["git", "-C", str(ROOT), "fetch", "--depth", "1", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", str(ROOT), "checkout", "--detach", COMMIT], check=True)
print("Ditto source:", subprocess.check_output(
    ["git", "-C", str(ROOT), "rev-parse", "HEAD"], text=True
).strip())


## 3 · Download only the required PyTorch checkpoints

This skips the much larger ONNX and TensorRT folders. The model repository is public; no Hugging Face token is needed.


In [ ]:
import subprocess, textwrap
from pathlib import Path

PYTHON = "/content/ditto-env/bin/python"
ROOT = Path("/content/ditto-talkinghead")
script = r"""
from pathlib import Path
from huggingface_hub import snapshot_download

target = Path('/content/ditto-talkinghead/checkpoints')
snapshot_download(
    repo_id='digital-avatar/ditto-talkinghead',
    revision='e4a2f60328ee7c32af585ac4b3cce299e4c8e254',
    local_dir=target,
    allow_patterns=[
        'ditto_cfg/v0.4_hubert_cfg_pytorch.pkl',
        'ditto_pytorch/aux_models/*',
        'ditto_pytorch/models/*',
    ],
    max_workers=4,
)

required = [
    target / 'ditto_cfg/v0.4_hubert_cfg_pytorch.pkl',
    target / 'ditto_pytorch/aux_models/hubert_streaming_fix_kv.onnx',
    target / 'ditto_pytorch/models/lmdm_v0.4_hubert.pth',
    target / 'ditto_pytorch/models/decoder.pth',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError('Missing checkpoint files: ' + ', '.join(missing))

total = sum(path.stat().st_size for path in target.rglob('*') if path.is_file())
print(f'Checkpoint download complete: {total / 1024**3:.2f} GiB')
"""
subprocess.run([PYTHON, "-c", textwrap.dedent(script)], check=True)


## 4 · Verify CUDA and libraries

`CUDAExecutionProvider` is preferred for the auxiliary ONNX models. If it is unavailable, ONNX Runtime can fall back to CPU, but generation will be slower.


In [ ]:
import subprocess
from pathlib import Path

PYTHON = "/content/ditto-env/bin/python"
tests = {
    "NVIDIA GPU": [
        "nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"
    ],
    "PyTorch CUDA": [
        PYTHON, "-c",
        "import torch; "
        "print('Torch:', torch.__version__); "
        "print('Built CUDA:', torch.version.cuda); "
        "print('CUDA available:', torch.cuda.is_available()); "
        "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')",
    ],
    "NumPy and OpenCV": [
        PYTHON, "-c",
        "import numpy, cv2; print('NumPy:', numpy.__version__); print('OpenCV:', cv2.__version__)",
    ],
    "MediaPipe": [PYTHON, "-c", "import mediapipe; print('MediaPipe:', mediapipe.__version__)"],
    "Librosa": [PYTHON, "-c", "import librosa; print('Librosa:', librosa.__version__)"],
    "ONNX Runtime": [
        PYTHON, "-c",
        "import torch, onnxruntime as ort; print('ORT:', ort.__version__); print('Providers:', ort.get_available_providers())",
    ],
    "Gradio": [PYTHON, "-c", "import gradio; print('Gradio:', gradio.__version__)"],
}

sections = []
failures = []
for title, command in tests.items():
    result = subprocess.run(command, text=True, capture_output=True)
    section = [f"{'=' * 64}", title, f"Return code: {result.returncode}"]
    if result.stdout.strip():
        section.extend(["OUTPUT:", result.stdout.strip()])
    if result.stderr.strip():
        section.extend(["ERROR:", result.stderr.strip()])
    sections.append("\n".join(section))
    if result.returncode:
        failures.append(title)

report = "\n\n".join(sections)
Path("/content/ditto_diagnostic.txt").write_text(report, encoding="utf-8")
print(report)
print("\n" + "=" * 64)
if failures:
    print("⚠️ Verification found failures:", ", ".join(failures))
    print("No exception was raised. Copy the failing section above or download /content/ditto_diagnostic.txt.")
else:
    print("✅ Environment verification passed. Continue to the Gradio app cells.")


## 5 · Create the Gradio app

The custom UI adds image/audio validation, audio conversion to mono 16 kHz WAV, eight expression choices, model reuse, one-job-at-a-time queuing, and an optional disclosure watermark.


In [ ]:
from pathlib import Path

APP_PATH = Path("/content/ditto-talkinghead/gradio_colab_app.py")
APP_CODE = 'from __future__ import annotations\n\nimport argparse\nimport gc\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport types\nimport uuid\nfrom pathlib import Path\n\nimport gradio as gr\nimport numpy as np\nimport librosa\nimport torch\nfrom PIL import Image, ImageOps\n\n# Load PyTorch\'s bundled CUDA/cuDNN libraries before ONNX Runtime.\ntry:\n    import onnxruntime as ort\n    if hasattr(ort, "preload_dlls"):\n        ort.preload_dlls()\n    print("ONNX Runtime providers:", ort.get_available_providers())\nexcept Exception as exc:\n    print("ONNX Runtime preload note:", exc)\n\nROOT = Path(__file__).resolve().parent\nCHECKPOINTS = ROOT / "checkpoints"\nDATA_ROOT = CHECKPOINTS / "ditto_pytorch"\nCFG_PATH = CHECKPOINTS / "ditto_cfg" / "v0.4_hubert_cfg_pytorch.pkl"\nOUTPUT_ROOT = ROOT / "outputs" / "gradio"\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\nos.chdir(ROOT)\n\n# apt normally supplies ffmpeg. imageio-ffmpeg is a portable fallback and the\n# PATH shim also covers Ditto\'s own final audio-mux command.\nFFMPEG_BIN = shutil.which("ffmpeg")\nif FFMPEG_BIN is None:\n    from imageio_ffmpeg import get_ffmpeg_exe\n    bundled_ffmpeg = Path(get_ffmpeg_exe()).resolve()\n    shim_dir = OUTPUT_ROOT / ".bin"\n    shim_dir.mkdir(parents=True, exist_ok=True)\n    shim = shim_dir / "ffmpeg"\n    if not shim.exists():\n        shim.symlink_to(bundled_ffmpeg)\n    os.environ["PATH"] = f"{shim_dir}:{os.environ.get(\'PATH\', \'\')}"\n    FFMPEG_BIN = str(shim)\n\n# Ditto ships a tiny Cython compositor. Prefer it, but retain a vectorized\n# NumPy fallback for Colab images where Python development headers are absent.\ntry:\n    from core.utils.blend import blend_images_cy as _blend_probe\nexcept Exception as exc:\n    print("Cython compositor unavailable; using NumPy fallback:", exc)\n    sys.modules.pop("core.utils.blend", None)\n    blend_module = types.ModuleType("core.utils.blend")\n\n    def blend_images_cy(mask_warped, frame_warped, frame_rgb, result):\n        alpha = mask_warped[..., None].astype(np.float32, copy=False)\n        blended = alpha * frame_warped + (1.0 - alpha) * frame_rgb\n        result[...] = np.clip(blended, 0, 255).astype(np.uint8)\n\n    blend_module.blend_images_cy = blend_images_cy\n    sys.modules["core.utils.blend"] = blend_module\n\nfrom inference import run, seed_everything\nfrom stream_pipeline_offline import StreamSDK\n\nEMOTION_MAP = {\n    "Angry": 0,\n    "Disgust": 1,\n    "Fear": 2,\n    "Happy": 3,\n    "Neutral": 4,\n    "Sad": 5,\n    "Surprise": 6,\n    "Contempt": 7,\n}\nMAX_AUDIO_SECONDS = 60.0\nMODEL_LOCK = threading.Lock()\n_SDK: StreamSDK | None = None\n\n\ndef get_sdk() -> StreamSDK:\n    global _SDK\n    if _SDK is None:\n        if not torch.cuda.is_available():\n            raise RuntimeError("A CUDA GPU is required. In Colab select Runtime → Change runtime type → T4 GPU.")\n        print("Loading Ditto PyTorch models (first request only)…")\n        _SDK = StreamSDK(str(CFG_PATH), str(DATA_ROOT))\n        print("Ditto models loaded.")\n    return _SDK\n\n\ndef clean_old_jobs(max_age_hours: float = 8.0) -> None:\n    cutoff = time.time() - max_age_hours * 3600\n    for item in OUTPUT_ROOT.iterdir():\n        try:\n            if item.is_dir() and item.stat().st_mtime < cutoff:\n                shutil.rmtree(item, ignore_errors=True)\n        except OSError:\n            pass\n\n\ndef run_checked(command: list[str]) -> None:\n    result = subprocess.run(command, text=True, capture_output=True)\n    if result.returncode:\n        tail = (result.stderr or result.stdout)[-3000:]\n        raise RuntimeError(f"Command failed ({result.returncode}):\\n{tail}")\n\n\ndef prepare_image(source_path: str, destination: Path) -> None:\n    Image.MAX_IMAGE_PIXELS = 40_000_000\n    with Image.open(source_path) as opened:\n        image = ImageOps.exif_transpose(opened).convert("RGB")\n        width, height = image.size\n        if min(width, height) < 256:\n            raise ValueError("Portrait is too small. Use an image at least 256 px on each side.")\n        if width * height > 40_000_000:\n            raise ValueError("Portrait is too large. Use an image below 40 megapixels.")\n        image.save(destination, format="PNG", optimize=True)\n\n\ndef prepare_audio(source_path: str, destination: Path) -> float:\n    run_checked([\n        FFMPEG_BIN, "-hide_banner", "-loglevel", "error", "-y",\n        "-i", source_path, "-vn", "-ac", "1", "-ar", "16000",\n        "-c:a", "pcm_s16le", str(destination),\n    ])\n    duration = float(librosa.get_duration(path=str(destination)))\n    if duration < 0.25:\n        raise ValueError("Audio is too short; provide at least 0.25 seconds.")\n    if duration > MAX_AUDIO_SECONDS:\n        raise ValueError(f"Audio is {duration:.1f}s. This Colab UI limits each job to {MAX_AUDIO_SECONDS:.0f}s.")\n    return duration\n\n\ndef add_disclosure_watermark(source: Path, destination: Path) -> bool:\n    font = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"\n    vf = (\n        "drawbox=x=w-238:y=h-52:w=226:h=40:color=black@0.55:t=fill,"\n        f"drawtext=fontfile={font}:text=\'AI-generated avatar\':"\n        "fontcolor=white:fontsize=18:x=w-tw-20:y=h-th-20"\n    )\n    result = subprocess.run([\n        FFMPEG_BIN, "-hide_banner", "-loglevel", "error", "-y",\n        "-i", str(source), "-vf", vf, "-c:v", "libx264",\n        "-preset", "veryfast", "-crf", "18", "-c:a", "copy",\n        "-movflags", "+faststart", str(destination),\n    ], text=True, capture_output=True)\n    if result.returncode:\n        print("Watermark step skipped:", result.stderr[-1200:])\n        return False\n    return True\n\n\ndef generate_avatar(\n    source_image: str,\n    driving_audio: str,\n    emotion: str,\n    sampling_steps: int,\n    max_output_dimension: int,\n    crop_scale: float,\n    seed: int,\n    add_watermark: bool,\n    consent_confirmed: bool,\n    progress=gr.Progress(),\n):\n    if not consent_confirmed:\n        raise gr.Error("Confirm that you have permission to use the face and audio.")\n    if not source_image:\n        raise gr.Error("Upload one clear portrait image.")\n    if not driving_audio:\n        raise gr.Error("Upload or record the driving speech audio.")\n\n    with MODEL_LOCK:\n        started = time.perf_counter()\n        clean_old_jobs()\n        job_dir = OUTPUT_ROOT / f"job_{time.strftime(\'%Y%m%d_%H%M%S\')}_{uuid.uuid4().hex[:8]}"\n        job_dir.mkdir(parents=True, exist_ok=False)\n        image_path = job_dir / "portrait.png"\n        audio_path = job_dir / "speech_16k.wav"\n        raw_path = job_dir / "avatar_raw.mp4"\n        final_path = job_dir / "avatar.mp4"\n\n        try:\n            progress(0.03, desc="Validating portrait and audio…")\n            prepare_image(source_image, image_path)\n            duration = prepare_audio(driving_audio, audio_path)\n\n            progress(0.10, desc="Loading models (first job takes longer)…")\n            sdk = get_sdk()\n            seed = max(0, int(seed))\n            seed_everything(seed)\n            if torch.cuda.is_available():\n                torch.cuda.reset_peak_memory_stats()\n\n            progress(0.18, desc="Animating the avatar…")\n            more_kwargs = {\n                "setup_kwargs": {\n                    "emo": EMOTION_MAP.get(emotion, 4),\n                    "sampling_timesteps": int(sampling_steps),\n                    "max_size": int(max_output_dimension),\n                    "crop_scale": float(crop_scale),\n                },\n                "run_kwargs": {},\n            }\n            run(sdk, str(audio_path), str(image_path), str(raw_path), more_kwargs)\n            if not raw_path.exists() or raw_path.stat().st_size == 0:\n                raise RuntimeError("Ditto finished without creating an output video.")\n\n            progress(0.92, desc="Finalizing video…")\n            watermarked = False\n            if add_watermark:\n                watermarked = add_disclosure_watermark(raw_path, final_path)\n            output_path = final_path if watermarked else raw_path\n\n            elapsed = time.perf_counter() - started\n            peak = (\n                torch.cuda.max_memory_allocated() / 1024**3\n                if torch.cuda.is_available() else 0.0\n            )\n            progress(1.0, desc="Done")\n            status = (\n                f"### ✅ Avatar ready\\n"\n                f"- Audio: **{duration:.1f}s** · Render: **{elapsed:.1f}s**\\n"\n                f"- Emotion: **{emotion}** · Seed: **{seed}** · Steps: **{int(sampling_steps)}**\\n"\n                f"- PyTorch peak VRAM: **{peak:.2f} GiB**"\n            )\n            if add_watermark and not watermarked:\n                status += "\\n- ⚠️ Disclosure watermark could not be added; the raw result is shown."\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            return str(output_path), status\n        except gr.Error:\n            raise\n        except Exception as exc:\n            traceback.print_exc()\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            raise gr.Error(\n                f"Generation failed: {exc}. If a worker error occurred, restart the Launch cell and try a frontal portrait."\n            ) from exc\n\n\nCSS = """\n.gradio-container {max-width: 1180px !important;}\n.hero {text-align:center; padding: 12px 0 4px;}\n.hero h1 {font-size: 2.25rem; margin-bottom: .25rem;}\n.notice {border-left: 4px solid #6366f1; padding: 10px 14px; background: rgba(99,102,241,.08); border-radius: 8px;}\n"""\n\nwith gr.Blocks(\n    theme=gr.themes.Soft(primary_hue="indigo", secondary_hue="violet"),\n    css=CSS,\n    title="Open Avatar Studio — Ditto",\n) as demo:\n    gr.HTML("""\n    <div class="hero">\n      <h1>🎭 Open Avatar Studio</h1>\n      <p>One portrait + speech audio → expressive talking-avatar video</p>\n    </div>\n    """)\n    gr.Markdown(\n        "<div class=\'notice\'><b>Backend:</b> Ant Group Ditto PyTorch (Apache‑2.0). "\n        "The first job loads the models; later jobs reuse them. Keep the private Gradio link to yourself.</div>"\n    )\n\n    with gr.Row(equal_height=False):\n        with gr.Column(scale=5):\n            source_image = gr.Image(\n                label="1 · Portrait image",\n                type="filepath",\n                sources=["upload", "webcam"],\n                height=390,\n            )\n            driving_audio = gr.Audio(\n                label="2 · Driving speech (max 60 seconds)",\n                type="filepath",\n                sources=["upload", "microphone"],\n            )\n            emotion = gr.Dropdown(\n                choices=list(EMOTION_MAP), value="Neutral", label="3 · Expression style"\n            )\n\n            with gr.Accordion("Advanced controls", open=False):\n                sampling_steps = gr.Slider(\n                    10, 50, value=20, step=10,\n                    label="Motion sampling steps",\n                    info="10 is faster; 30–50 may improve difficult clips.",\n                )\n                max_output_dimension = gr.Radio(\n                    choices=[("Fast · 720 px", 720), ("Balanced · 1080 px", 1080), ("Source up to 1920 px", 1920)],\n                    value=1080,\n                    label="Maximum output dimension",\n                )\n                crop_scale = gr.Slider(\n                    1.8, 3.2, value=2.3, step=0.1,\n                    label="Face crop scale",\n                    info="Increase if the head is clipped; 2.3 is the model default.",\n                )\n                seed = gr.Number(value=1024, precision=0, label="Seed")\n                add_watermark = gr.Checkbox(\n                    value=True, label="Add ‘AI-generated avatar’ disclosure watermark"\n                )\n\n            consent = gr.Checkbox(\n                value=False,\n                label="I have permission to use this face and audio, and I will disclose synthetic media where appropriate.",\n            )\n            generate_button = gr.Button("✨ Generate avatar", variant="primary", size="lg")\n\n            example_image = ROOT / "example" / "image.png"\n            example_audio = ROOT / "example" / "audio.wav"\n            if example_image.exists() and example_audio.exists():\n                gr.Examples(\n                    examples=[[str(example_image), str(example_audio)]],\n                    inputs=[source_image, driving_audio],\n                    label="Official Ditto sample",\n                )\n\n        with gr.Column(scale=6):\n            output_video = gr.Video(label="Generated avatar", height=520)\n            status = gr.Markdown("Upload inputs, confirm permission, then generate.")\n\n    generate_button.click(\n        fn=generate_avatar,\n        inputs=[\n            source_image,\n            driving_audio,\n            emotion,\n            sampling_steps,\n            max_output_dimension,\n            crop_scale,\n            seed,\n            add_watermark,\n            consent,\n        ],\n        outputs=[output_video, status],\n        api_name=False,\n    )\n\n    gr.Markdown(\n        "---\\n**Best input:** one front-facing person, visible shoulders, even light, clear speech, and minimal background noise. "\n        "Do not use this tool for impersonation, fraud, harassment, or non-consensual media. "\n        "[Ditto source](https://github.com/antgroup/ditto-talkinghead) · "\n        "[Apache‑2.0 license](https://github.com/antgroup/ditto-talkinghead/blob/main/LICENSE)"\n    )\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--host", default="0.0.0.0")\n    parser.add_argument("--port", type=int, default=7860)\n    parser.add_argument("--share", action="store_true")\n    args = parser.parse_args()\n    demo.queue(max_size=4, default_concurrency_limit=1).launch(\n        server_name=args.host,\n        server_port=args.port,\n        share=args.share,\n        show_error=True,\n        allowed_paths=[str(OUTPUT_ROOT), str(ROOT / "example")],\n    )\n\n\nif __name__ == "__main__":\n    main()\n'
APP_PATH.write_text(APP_CODE, encoding="utf-8")
print(f"Wrote {APP_PATH} ({APP_PATH.stat().st_size:,} bytes)")


## 6 · Launch the studio

Run this cell and open the printed `https://....gradio.live` link. **Do not share that URL**—anyone with it can submit jobs to your GPU.

Stop the cell when finished. Generated files remain under `/content/ditto-talkinghead/outputs/gradio/` until the Colab runtime is deleted.


In [ ]:
import subprocess

command = [
    "/content/ditto-env/bin/python", "-u",
    "/content/ditto-talkinghead/gradio_colab_app.py",
    "--host", "0.0.0.0", "--port", "7860", "--share",
]
print("Launching Gradio… the first generation will load models.")
subprocess.run(command, check=True)


## Usage tips / वापरण्याच्या सूचना

1. Upload a clear, front-facing portrait with one visible face.
2. Upload speech audio or record from the microphone (up to 60 seconds per job).
3. Choose an expression; **Neutral** or **Happy** are good starting points.
4. Confirm permission and click **Generate avatar**.
5. Download the MP4 from the video player.

**If face detection fails:** use a less tightly cropped image, even lighting, and a face at least ~256 px tall.  
**If CUDA runs out of memory:** choose 720 px, stop other GPU notebooks, then restart the Launch cell.  
**If the Gradio link expires:** rerun only the Launch cell.  
**Need a generated/cloned voice first?** Use this repository's `Voicebox_Colab.ipynb`, download its WAV, and upload that WAV here.
